# Assignment 1 — Optimizers for Neural Network Training (and Optional Hyperparameter Tuning)

## Goal
In this assignment you will:
- train a small MLP **from scratch (NumPy)**, without PyTorch/TensorFlow autograd,
- understand and implement common **optimizers** (SGD, Momentum, Adam),
- run controlled experiments and compare convergence behavior and generalization,
- *(optional)* perform **hyperparameter tuning** using a validation split.

## What you submit
1. This notebook with all requested code cells filled in.
2. Fill in the blanks in Latex portions with the correct equations and update rules.

## Rules / constraints
- Use only **NumPy** (already imported). No deep learning frameworks.
- Keep experiments reproducible: set random seeds when asked.
- When comparing optimizers, keep the **model architecture** and **data split** fixed unless the question explicitly asks otherwise.

---

## Quick notation
Let a dataset be $\{(x_i, y_i)\}_{i=1}^N$
Let the model be $f(x;\theta)$ with parameters $\theta$.  
Training minimizes the empirical risk:
$$
\min_{\theta}\; \mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\ell\big(f(x_i;\theta), y_i\big)
$$
For this notebook we use MSE loss:
$$
\ell(\hat y, y) = \|\hat y - y\|_2^2
$$


In [ ]:
import numpy as np

def set_seed(seed=42):
    np.random.seed(seed)

def train_test_split(X, y, test_ratio=0.2):
    n = X.shape[0]
    idx = np.random.permutation(n)
    n_test = int(n * test_ratio)
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]


## Part A — Minimal NN components (Parameters, Layers)

We implement a tiny neural network engine:
- **`Parameter`** stores a trainable tensor and its gradient.
- Each **`Layer`** implements:
  - forward pass: $y = \text{layer}(x)$
  - backward pass: given $\frac{\partial \mathcal{L}}{\partial y}$, compute $\frac{\partial \mathcal{L}}{\partial x}$ and accumulate parameter gradients.



In [ ]:
class Parameter:
    """
    Holds trainable weights/biases and their gradients.
    """
    def __init__(self, data):
        self.data = data.astype(np.float64)
        self.grad = np.zeros_like(self.data)

    def zero_grad(self):
        self.grad[...] = 0.0


### Backprop interface

For a layer with input $x$ and output $y$:
$$
y = g(x)
$$
During backprop we receive $\frac{\partial \mathcal{L}}{\partial y}$ and must compute:
$$
\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y}\cdot \frac{\partial y}{\partial x}
$$
Layers that have parameters (e.g. Linear) must also accumulate:
$$
\frac{\partial \mathcal{L}}{\partial W},\;\frac{\partial \mathcal{L}}{\partial b}
$$

**Implementation note:** in this notebook, each layer caches what it needs from the forward pass to compute gradients in the backward pass.


In [ ]:
class Layer:
    def forward(self, x):
        raise NotImplementedError

    def backward(self, grad_out):
        raise NotImplementedError

    def parameters(self):
        return []

    def __call__(self, x):
        # "when passed input the class object forward should be called"
        return self.forward(x)


## Part B — MLP building blocks

### Linear layer
A fully connected layer computes:
$$
y = xW + b
$$
If $x\in\mathbb{R}^{N\times d_{in}}$ and $W\in\mathbb{R}^{d_{in}\times d_{out}}$, then $y\in\mathbb{R}^{N\times d_{out}}$.

Gradients used in backprop:
$$
\frac{\partial \mathcal{L}}{\partial W} = x^\top \frac{\partial \mathcal{L}}{\partial y},\qquad
\frac{\partial \mathcal{L}}{\partial b} = \sum_{i=1}^{N}\frac{\partial \mathcal{L}}{\partial y_i},\qquad
\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y}W^\top
$$

### Nonlinearities
We will use elementwise activations such as:
- $\tanh(z)$
- $\mathrm{ReLU}(z)=\max(0,z)$
- $\sigma(z)=\frac{1}{1+e^{-z}}$



In [ ]:
class Linear(Layer):
    def __init__(self, in_features, out_features, bias=True):
        # Your code goes here




    def forward(self, x):
        # Your code goes here




    def backward(self, grad_out):
        # Your code goes here


        return grad_x

    def parameters(self):
        # Your code goes here


class Tanh(Layer):
    def __init__(self):
        self.y_cache = None

    def forward(self, x):
      # Your code goes here



    def backward(self, grad_out):
        # Your code goes here



class ReLU(Layer):
    def __init__(self):
        self.mask = None

    def forward(self, x):
        # Your code goes here



    def backward(self, grad_out):
        # Your code goes here




class Sigmoid(Layer):
    def __init__(self):
        self.y_cache = None

    def forward(self, x):
        # Your code goes here



    def backward(self, grad_out):
        # Your code goes here




class MSELoss:
    """
    Mean Squared Error loss.
    forward(pred, target) returns scalar loss
    backward() returns dLoss/dPred with shape like pred
    """
    def __init__(self):
        self.pred_cache = None
        self.tgt_cache = None

    def forward(self, pred, target):
        # Your code goes here



    def backward(self):
        # Your code goes here





### The MLP container

An MLP is a composition of layers:
$$
f(x;\theta) = (g_L\circ g_{L-1}\circ \cdots \circ g_1)(x)
$$
Backprop simply applies the chain rule in reverse order.

**Important:** This notebook does not use an automatic differentiation library. Your backward implementations must be correct for training to work.


In [ ]:
class MLP:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)  # uses __call__ -> forward
        return x

    def backward(self, grad_out):
        for layer in reversed(self.layers):
            grad_out = layer.backward(grad_out)
        return grad_out

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

    def zero_grad(self):
        for p in self.parameters():
            p.zero_grad()

    def __call__(self, x):
        return self.forward(x)


## Part C — Data

We generate a synthetic regression dataset:
$$
y = \sin(1.5x_1) + 0.5\cos(2x_2) + 0.3x_1x_2 + \epsilon,\qquad \epsilon\sim\mathcal{N}(0,\sigma^2)
$$




In [ ]:
def make_noisy_data(n=2048, noise_std=0.1):
    """
    y = f(x1,x2) + noise
    Choose a function that's nonlinear and smooth.
    """
    X = np.random.uniform(low=-2.0, high=2.0, size=(n, 2)).astype(np.float64)
    x1 = X[:, 0:1]
    x2 = X[:, 1:2]
    # Nonlinear target
    y_clean = np.sin(1.5 * x1) + 0.5 * np.cos(2.0 * x2) + 0.3 * x1 * x2
    y = y_clean + np.random.randn(n, 1) * noise_std
    return X, y


## Part D — Optimizers

An optimizer updates parameters $\theta$ using gradients $\nabla_\theta \mathcal{L}$.

### D.1: Vanilla SGD
$$
\theta_{t+1} = \text{_________},\qquad g_t=\text{_________}
$$
where $\eta>0$ is the learning rate.

### D.2: Weight decay (L2 regularization)
A common implementation adds:
$$
g_t \leftarrow \text{_________}
$$
leading to:
$$
\theta_{t+1} = \text{_________}
$$
This shrinks weights and can improve generalization.

### D.3: Momentum
Maintain a velocity $v_t$:
$$
v_{t} = \text{_________},\qquad \theta_{t+1}=\text{_________}
$$
where $\mu\in[0,1)$ is the momentum coefficient.

### D.4: Adam
Adam keeps exponential moving averages of gradients and squared gradients:
$$
m_t= \text{_________},\qquad
v_t=\text{_________}
$$
Bias correction:
$$
\hat m_t = \text{_________}, \qquad \hat v_t = \text{_________}
$$
Update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \text{________________}
$$

---

### Tasks (core)
1. Read the optimizer implementations carefully (SGD, Momentum, Adam).
2. Run experiments comparing these optimizers under controlled settings.


---

### Optional extension
Add *new* optimizer classes under this cell (below the provided ones) and test them:
- **RMSProp**
  $$
  v_t=\text{_________},\qquad \theta_{t+1}=\text{_________}
  $$
- **Nesterov momentum** (one common form)
  $$
  v_t=\text{_________},\qquad \theta_{t+1}=\text{_________}
  $$
- **AdamW** (decoupled weight decay)
  $$
  \theta_{t+1}=\text{_________}
  $$

If you implement any optional optimizer, clearly label it and include results in the comparison section.


In [ ]:
class Optimizer:
    def __init__(self, params, lr=1e-3):
        self.params = params
        self.lr = float(lr)

    def step(self):
        raise NotImplementedError

    def zero_grad(self):
        for p in self.params:
            p.zero_grad()


class SGD(Optimizer):
    def __init__(self, params, lr=1e-3, weight_decay=0.0):
        # Your code goes here

    def step(self):
        # Your code goes here


class SGD_Momentum(Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.9, weight_decay=0.0):
        # Your code goes here

    def step(self):
        # Your code goes here


class Adam(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        # Your code goes here


    def step(self):
        # Your code goes here



## Part E — Training loop and evaluation

We train with mini-batches. For each batch:
1. Forward pass
2. Compute loss
3. Backward pass (compute gradients)
4. Optimizer step (update parameters)
5. Zero gradients

### MSE gradient check (sanity)
For MSE averaged over all entries:
$$
\mathcal{L}=\frac{1}{|\hat y|}\sum (\hat y - y)^2
\quad\Rightarrow\quad
\frac{\partial \mathcal{L}}{\partial \hat y}=\frac{2}{|\hat y|}(\hat y-y)
$$
This notebook implements that scaling.

---

### Task (recommended instrumentation)
Modify the training function (or write a new one) to **record**:
- per-epoch training loss (mean over batches)
- per-epoch test loss

Then plot curves for different optimizers on the **same axes**.


In [ ]:
def iterate_minibatches(X, y, batch_size=64, shuffle=True):
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, n, batch_size):
        batch_idx = idx[start:start + batch_size]
        yield X[batch_idx], y[batch_idx]


def train(model, optimizer, X_train, y_train, X_test, y_test, epochs=200, batch_size=64, print_every=20):
    loss_fn = MSELoss()

    for epoch in range(1, epochs + 1):
        # train
        model.zero_grad()
        train_losses = []

        for xb, yb in iterate_minibatches(X_train, y_train, batch_size=batch_size, shuffle=True):
            # Your code goes here



        # eval
        # Your code goes here



        if epoch % print_every == 0 or epoch == 1:
            print(f"Epoch {epoch:4d} | train_loss={np.mean(train_losses):.6f} | test_loss={test_loss:.6f}")


## Part F — Experiments: optimizer comparison (required)

You will run controlled comparisons. Keep everything fixed except the optimizer (or the specific hyperparameter under study).

### F.1 Baseline configuration
Use:
- same random seed
- same train/test split
- same model architecture
- same batch size and number of epochs

### Required experiments
Run at least **three** optimizers among:
- SGD
- SGD + Momentum
- Adam
(+ any optional optimizer you implemented)

Experimental Setup: **Sensitivity to Initialization** For each optimizer (SGD, Momentum, Adam), you must perform the training starting from three different initializations (random seeds).

- Record the convergence path and final loss for each starting point.

- Analysis: Note down how stable each optimizer is. Does the choice of starting point significantly change the final test loss or the speed of convergence?



---

## Part G — Hyperparameter tuning (optional but encouraged)

Use a validation split: split train into **train/val**, and use **test only once at the end**.

### G.1 What to tune
Choose a search space, e.g.
- learning rate $\eta \in \{10^{-4},10^{-3},10^{-2},10^{-1}\}$
- weight decay $\lambda \in \{0,10^{-5},10^{-4},10^{-3}\}$
- hidden width $h \in \{8, 20, 64\}$
- activation $\in \{\tanh,\mathrm{ReLU}\}$
- momentum $\mu \in \{0.5, 0.9, 0.99\}$ (if using momentum)

### G.2 Search method
- **Grid search:** try all combinations (can be expensive).
- **Random search:** sample configurations; often works well in practice.

Pseudo-code:
$
\text{for }k=1..K:\\
\text{sample }(\eta,\lambda,\dots)\\
\text{train on train}\\
\text{evaluate on val}\\
\text{keep best}
$

### If you do this optional part, include:
- your search space
- number of trials \(K\)
- best configuration (val loss)
- final test loss after retraining using train+val (optional) with the best hyperparameters




In [ ]:
# Example

set_seed(0)

X, y = make_noisy_data(n=3000, noise_std=0.10)
X_train, y_train, X_test, y_test = train_test_split(X, y, test_ratio=0.2)

# Example: 2 -> 4 -> 1 network (like your 2*4*1)
model = MLP([
    Linear(2, 2),
    Sigmoid(),
    Linear(2, 2),
    Sigmoid(),
    Linear(2, 1),
])

# Choose ONE optimizer:
# opt = SGD(model.parameters(), lr=1e-2, weight_decay=1e-4)
# opt = SGD_Momentum(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)


# Your code goes here


In [ ]:
seeds = [0, 42, 123]
optimizers_to_test = ['SGD', 'Momentum', 'Adam']

for opt_name in optimizers_to_test:
    for s in seeds:
        set_seed(s) # Ensure different starting weights
        # Initialize model and optimizer here
        # Run training loop
        # Store/Plot results

        # Your code goes here

Different activation functions have different "inductive biases." Tanh is smooth and zero-centered, while ReLU is piecewise linear and can lead to "dead" neurons.

Fill in the derivative used for backpropagation through a ReLU layer:$$\frac{\partial \text{ReLU}(z)}{\partial z} = \begin{cases} \text{____} & \text{if } z > 0 \\ \text{____} & \text{if } z \le 0 \end{cases}$$

Analysis Question:
After running the code below, compare the "smoothness" of the resulting function approximation between Tanh and ReLU. Which one looks more like the original synthetic function?

# Plots

In [ ]:
import matplotlib.pyplot as plt

def true_function(X):
    """
    Must match make_noisy_data()'s clean function (without noise).
    X: (N,2)
    returns: (N,1)
    """
    x1 = X[:, 0:1]
    x2 = X[:, 1:2]
    return np.sin(1.5 * x1) + 0.5 * np.cos(2.0 * x2) + 0.3 * x1 * x2


def plot_slice_true_vs_model(model, fix="x2", fixed_value=0.0,
                             sweep_min=-2.0, sweep_max=2.0, n_points=1000,
                             title=None):
    """
    fix: "x1" or "x2"
    fixed_value: value to hold the fixed variable at
    Sweeps the other variable and plots y_true vs y_model.
    """
    sweep = np.linspace(sweep_min, sweep_max, n_points).reshape(-1, 1)

    if fix == "x2":
        X = np.hstack([sweep, np.full_like(sweep, fixed_value)])  # x1 varies, x2 fixed
        x_axis = sweep[:, 0]
        x_label = "x1 (x2 fixed)"
    elif fix == "x1":
        X = np.hstack([np.full_like(sweep, fixed_value), sweep])  # x2 varies, x1 fixed
        x_axis = sweep[:, 0]
        x_label = "x2 (x1 fixed)"
    else:
        raise ValueError("fix must be 'x1' or 'x2'")

    y_true = true_function(X)
    y_pred = model(X)

    plt.figure(figsize=(12, 7))
    plt.plot(x_axis, y_true[:, 0], label="True function")
    plt.plot(x_axis, y_pred[:, 0], label="Neural net")
    plt.xlabel(x_label + f" = {fixed_value:.3f}" if fix in ["x1", "x2"] else x_label)
    plt.ylabel("y")
    if title is None:
        title = f"Slice plot: fix {fix} = {fixed_value}"
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
# Plot
plot_slice_true_vs_model(model, fix="x2", fixed_value=0.0)
plot_slice_true_vs_model(model, fix="x2", fixed_value=1.0)
plot_slice_true_vs_model(model, fix="x1", fixed_value=-1.0)
plot_slice_true_vs_model(model, fix="x1", fixed_value=1.3)


# Section II- Different Optimisers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define functions
def f1(x): return (x - 1.5)**2 + 0.5
def f2(x): return x**4 - 4*x**2 + 2
def f3(x): return 0.15 * (x**2) + np.sin(3.0*x) + 0.2 * np.sin(9.0*x)

# Generate domains
x1 = np.linspace(-10, 10, 400)
x2 = np.linspace(-3, 3, 400)
x3 = np.linspace(-5.12, 5.12, 1000)

# Create plots
fig, axs = plt.subplots(3, 1, figsize=(8, 12))
fig.subplots_adjust(hspace=0.4)

# Plot 1: Convex
axs[0].plot(x1, f1(x1), 'b-', linewidth=2)
axs[0].set_title('Convex Function (Parabola)')
axs[0].set_ylabel('$f_1(x)$')
axs[0].grid(True, alpha=0.3)

# Plot 2: Non-Convex
axs[1].plot(x2, f2(x2), 'r-', linewidth=2)
axs[1].set_title('Non-Convex Function')
axs[1].set_ylabel('$f_2(x)$')
axs[1].grid(True, alpha=0.3)

# Plot 3: Highly Curved
axs[2].plot(x3, f3(x3), 'g-', linewidth=2)
axs[2].set_title('Highly Curved Function')
axs[2].set_xlabel('x')
axs[2].set_ylabel('$f_3(x)$')
axs[2].grid(True, alpha=0.3)

plt.show()

In [ ]:
# --- Function 1: Shifted Convex ---
def df1(x):
    """
    Derivative of f1(x) = (x - 1.5)**2 + 0.5
    """
    return # TODO

# --- Function 2: Double Well ---
def df2(x):
    """
    Derivative of f2(x) = x**4 - 4*x**2 + 2
    """
    return # TODO

# --- Function 3: Ripples with Trend ---
def df3(x):
    """
    Derivative of f3(x) = 0.15x^2 + sin(3x) + 0.2sin(9x)
    """
    return # TODO

In [ ]:
import numpy as np

class Optimizer:
    def __init__(self, lr=0.01, weight_decay=0.0):
        self.lr = lr
        self.weight_decay = weight_decay

    def step(self, x, grad):
        raise NotImplementedError

class SGD(Optimizer):
    def __init__(self, lr=0.01, weight_decay=0.0):
        # Your code goes here

    def step(self, x, grad):
        # Your code goes here

class Momentum(Optimizer):
    # Your code goes here

    def step(self, x, grad):
        # Your code goes here

class Adam(Optimizer):
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=0.0):
        # Your code goes here

    def step(self, x, grad):
        # Your code goes here

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_optimization(func, x_range, histories, labels, title="Optimization Journey", frames=None):
    """
    Animates the optimization path of multiple optimizers on a 1D function.

    Args:
        func (callable): The objective function f(x) to plot in the background.
        x_range (tuple): (min_x, max_x) for the plot domain.
        histories (list of arrays): A list where each element is a sequence of x values
                                    visited by an optimizer.
        labels (list of str): Names for the legend (e.g., ['SGD', 'Momentum']).
        title (str): Title of the plot.
        frames (int, optional): Number of frames. Defaults to the length of the longest history.

    Returns:
        HTML object: The animation rendered as HTML/JS for Colab.
    """

    # 1. Setup the plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Generate background curve
    x_vals = np.linspace(x_range[0], x_range[1], 500)
    y_vals = func(x_vals)
    ax.plot(x_vals, y_vals, 'k--', alpha=0.3, label='Loss Landscape')

    # Set plot limits and labels
    ax.set_xlim(x_range)
    # Add some padding to y-axis
    y_min, y_max = y_vals.min(), y_vals.max()
    margin = (y_max - y_min) * 0.1
    ax.set_ylim(y_min - margin, y_max + margin)

    ax.set_title(title)
    ax.set_xlabel('Parameter x')
    ax.set_ylabel('Loss f(x)')
    ax.grid(True, alpha=0.3)

    # 2. Initialize plot elements for each optimizer
    # We use a scatter plot for the current point and a line for the trail
    lines = []
    points = []
    colors = plt.cm.jet(np.linspace(0, 0.9, len(histories))) # Distinct colors

    for label, color in zip(labels, colors):
        line, = ax.plot([], [], '-', color=color, alpha=0.6, linewidth=1.5)
        point, = ax.plot([], [], 'o', color=color, label=label, markersize=8)
        lines.append(line)
        points.append(point)

    ax.legend(loc='upper right')

    # 3. Determine number of frames
    if frames is None:
        frames = max(len(h) for h in histories)

    # 4. Update function for animation
    def update(frame):
        for i, history in enumerate(histories):
            # Handle cases where some optimizers finish earlier than others
            curr_idx = min(frame, len(history) - 1)

            # Get data up to current frame
            path_x = history[:curr_idx+1]
            path_y = func(np.array(path_x))

            # Update trail (line)
            lines[i].set_data(path_x, path_y)

            # Update current position (dot)
            points[i].set_data([path_x[-1]], [path_y[-1]])

        return lines + points

    # 5. Create Animation
    anim = FuncAnimation(fig, update, frames=frames, interval=100, blit=True)

    # Close the static plot to prevent double display in notebooks
    plt.close()

    # Return HTML object for Colab display
    return HTML(anim.to_jshtml())

In [ ]:
def run_and_animate(target_func, derivative_func, optimizer_cls, start_x, lr=0.1, n_steps=50, x_range=(-5,5), title=None, **opt_kwargs):
    """
    Runs an optimizer on a function and animates the result.

    Args:
        target_func (callable): The objective function f(x).
        derivative_func (callable): The derivative f'(x).
        optimizer_cls (class): The class of the optimizer (e.g., SGD, Adam).
                               Note: Pass the class, not an instance.
        start_x (float): The starting coordinate.
        lr (float): Learning rate.
        n_steps (int): Number of steps to run.
        x_range (tuple): (min, max) for the plot visualization.
        title (str): Optional title for the plot.
        **opt_kwargs: Additional arguments for the optimizer (e.g., beta=0.9).

    Returns:
        HTML: The interactive animation.
    """

    # 1. Instantiate the optimizer
    # We pass lr and any other specific arguments (like beta or epsilon) here
    optimizer = optimizer_cls(lr=lr, **opt_kwargs)

    # 2. Run the optimization loop
    history = []
    x = start_x
    history.append(x)

    """
    # Your Code goes Here
    """

    # 3. Convert history to numpy array
    history = np.array(history)

    # 4. Define labels and title
    opt_name = optimizer_cls.__name__
    if title is None:
        title = f"{opt_name} on Custom Function"
    label_str = f"{opt_name} (lr={lr})"

    # 5. Call the animation helper (defined in previous step)
    # We wrap history and label in lists because the animator expects lists of optimizers
    return animate_optimization(
        func=target_func,
        x_range=x_range,
        histories=[history],
        labels=[label_str],
        title=title
    )